In [0]:
from pyspark.sql import DataFrame

# lit used to add a constant value in a df
from pyspark.sql.functions import explode, col, lit
 
# StrucType - defines overall layout of the table
# StructField - defines a column
# StringType - defines a specific data inside that column
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
# use when csv have header
def isIgnoreHeader(file: str, validation: str, delimiter: str, textQualifier: str) -> DataFrame:

    # multiline - true means read the file across multiple line. 
    fullSchema = spark.read.format("json").option("multiline", "true").load(validation)
    parsedSchema = fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE")

    # get the column name and strips it
    header = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]
    
    # loop the columns and create a schema
    fields = [StructField(fieldName, StringType(), nullable=True) for fieldName in header]
    schema = StructType(fields)

    dfFile1 = spark.read.format("csv").schema(schema).option("header", False).option("delimiter", delimiter).option("quote", textQualifier).load(file)

    firstline = dfFile1.first()

    # loop the rows and drop the header
    dfFile = dfFile1.filter(lambda row: row != firstline) 

    return dfFile

In [0]:
# use when csv don't have header
def withoutHeader(file: str, validation: str, delimiter: str, textQualifier: str) -> DataFrame:

    fullSchema = spark.read.format("json").option("multiline", "true").load(validation)
    parsedSchema = fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE")

    header = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]
    fields = [StructField(fieldName, StringType(), nullable=True) for fieldName in header]
    schema = StructType(fields)

    dfFile = spark.read.format("csv").schema(schema).option("header", False).option("delimiter", delimiter).option("quote", textQualifier).load(file)

    return dfFile

In [0]:
def delimitedFile(file: str, validation: str, header: str, delimiter: str, textQualifier: str) -> DataFrame:

    # Read csv with all columns first (let spark infer structure)
    df_raw = spark.read.format("csv") \
        .option("header", header) \
        .option("delimiter", delimiter) \
        .option("quote", textQualifier) \
        .load(file)

    # drop TEMPLATE column if it exists
    if "TEMPLATE" in df_raw.columns:
        df_raw = df_raw.drop("TEMPLATE")

    # load schema definition (excluding TEMPLATE)
    fullSchema = spark.read.format("json").option("multiline", "true").load(validation)
    parsedSchema = fullSchema.select(explode(col("columnNames"))).select(col("col.FieldName"), col("col.DataType")).filter(col("col.FieldName") != "TEMPLATE")

    schemaHeader = [row[0].strip() for row in parsedSchema.select("FieldName").collect()]

    # select columns in schema order (add missing columns as NULL)
    select_exprs = []
    for col_name in schemaHeader:
        if col_name in df_raw.columns:
            select_exprs.append(col(col_name).cast(StringType()))
        else:
            select_exprs.append(lit(None).cast(StringType()).alias(col_name))

    dfFile = df_raw.select(select_exprs)

    return dfFile

In [0]:
def path_exists(pathToCheck: str) -> bool:

    # References the JVM gateway on the cluster via PySpark sparkContext to securely invoke the Hadoop FileSystem APIs
    sc = spark.sparkContext
    path_class = sc._gateway.jvm.org.apache.hadoop.fs.Path
    fs_class = sc._gateway.jvm.org.apache.hadoop.fs.FileSystem

    fs = fs_class.get(sc._jsc.hadoopConfiguration())
    IsExists = fs.exists(path_class(pathToCheck))
    
    return IsExists